In [7]:
!pip install -q ultralytics opencv-python filterpy pandas pyarrow yt-dlp

In [26]:
from pathlib import Path

sample = Path("data/videos/sample.mp4")
if sample.exists():
    sample.unlink()
    print("Removed sample.mp4")

Removed sample.mp4


In [27]:
from pathlib import Path
print(list(Path("data/videos").glob("*.mp4")))

[PosixPath('data/videos/Drone Tracking 1.mp4'), PosixPath('data/videos/Drone tracking 2.mp4')]


In [30]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO


trained_weights = "runs/detect/train/weights/best.pt"
data_yaml = "data/drone_yolo/drone_dataset.yaml"


if Path(data_yaml).exists() and not Path(trained_weights).exists():
    print("Training drone model (quick run)...")

    model = YOLO("yolo11n.pt")

    model.train(
        data=data_yaml,
        epochs=3,      # fast training
        imgsz=640,
        batch=8
    )


if Path(trained_weights).exists():
    print("Using TRAINED drone model")
    weights_path = trained_weights
else:
    print("Training skipped or failed → using default YOLO ")
    weights_path = "yolo11n.pt"


CFG = {
    "videos_dir": "data/videos",
    "output_videos_dir": "outputs/videos",
    "hf_dataset_dir": "outputs/hf_dataset",
    "weights_path": weights_path
}


Path(CFG["videos_dir"]).mkdir(parents=True, exist_ok=True)
Path(CFG["output_videos_dir"]).mkdir(parents=True, exist_ok=True)
Path(CFG["hf_dataset_dir"]).mkdir(parents=True, exist_ok=True)

print("Model being used:", CFG["weights_path"])

Training skipped or failed → using default YOLO 
Model being used: yolo11n.pt


In [31]:
from pathlib import Path

video_files = list(Path("data/videos").glob("*.mp4"))

if not video_files:
    print("No video found — creating dummy video")

    dummy_path = "data/videos/dummy.mp4"
    out = cv2.VideoWriter(dummy_path, cv2.VideoWriter_fourcc(*'mp4v'), 10, (640, 480))

    for _ in range(60):
        frame = np.zeros((480, 640, 3), dtype=np.uint8)
        cv2.putText(frame, "Dummy Video", (150,240),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
        out.write(frame)

    out.release()

print("Videos found:", list(Path("data/videos").glob("*.mp4")))

Videos found: [PosixPath('data/videos/Drone Tracking 1.mp4'), PosixPath('data/videos/Drone tracking 2.mp4')]


In [32]:


import pandas as pd
from pathlib import Path


if "dataset_rows" not in globals():
    dataset_rows = []


if len(dataset_rows) == 0:
    print("No detections found — creating dummy dataset")

    dataset_rows = [{
        "video": "sample.mp4",
        "frame": 0,
        "bbox": [0, 0, 100, 100],
        "confidence": 0.0
    }]

# save dataset
Path("outputs/hf_dataset").mkdir(parents=True, exist_ok=True)

df = pd.DataFrame(dataset_rows)
parquet_path = "outputs/hf_dataset/detections.parquet"
df.to_parquet(parquet_path, index=False)

print("Dataset saved at:", parquet_path)

Dataset saved at: outputs/hf_dataset/detections.parquet


In [35]:
from collections import deque
from pathlib import Path
import cv2
import numpy as np
from ultralytics import YOLO
from filterpy.kalman import KalmanFilter


model = YOLO(CFG["weights_path"])


dataset_rows = []


video_files = sorted(
    [p for p in Path(CFG["videos_dir"]).glob("*.mp4") if p.name.lower() != "sample.mp4"]
)

print("Videos being processed:")
for v in video_files:
    print("-", v.name)

def make_kf(x, y):
    kf = KalmanFilter(dim_x=4, dim_z=2)
    # state: [x, y, vx, vy]
    kf.x = np.array([[x], [y], [0.], [0.]])
    kf.F = np.array([
        [1., 0., 1., 0.],
        [0., 1., 0., 1.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]
    ])
    kf.H = np.array([
        [1., 0., 0., 0.],
        [0., 1., 0., 0.]
    ])
    kf.P *= 500.0
    kf.R = np.array([
        [15., 0.],
        [0., 15.]
    ])
    kf.Q = np.array([
        [1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 10., 0.],
        [0., 0., 0., 10.]
    ])
    return kf

for video_path in video_files:
    cap = cv2.VideoCapture(str(video_path))

    fps = int(cap.get(cv2.CAP_PROP_FPS) or 10)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out_path = Path(CFG["output_videos_dir"]) / f"{video_path.stem}_tracked.mp4"
    out = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    frame_idx = 0
    trajectory = deque(maxlen=100)

    kf = None
    missed_frames = 0
    max_missed_frames = 10
    last_bbox = None
    active_track = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        detections = []
        results = model(frame, verbose=False)

        for r in results:
            if r.boxes is None or len(r.boxes) == 0:
                continue
            for box in r.boxes:
                conf = float(box.conf[0])
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)
                detections.append((conf, x1, y1, x2, y2, cx, cy))


        best_det = max(detections, key=lambda d: d[0]) if detections else None

        if kf is not None:
            kf.predict()

        if best_det is not None:
            conf, x1, y1, x2, y2, cx, cy = best_det

            if kf is None:
                kf = make_kf(cx, cy)
            else:
                z = np.array([[cx], [cy]])
                kf.update(z)

            missed_frames = 0
            active_track = True
            last_bbox = (x1, y1, x2, y2, conf)

        elif kf is not None and active_track:
            missed_frames += 1
            if missed_frames > max_missed_frames:
                active_track = False
                last_bbox = None


        if kf is not None and active_track:
            px = int(kf.x[0, 0])
            py = int(kf.x[1, 0])
            trajectory.append((px, py))

            annotated = frame.copy()


            if last_bbox is not None:
                x1, y1, x2, y2, conf = last_bbox
                cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    annotated,
                    f"drone {conf:.2f}",
                    (x1, max(20, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2
                )


            cv2.circle(annotated, (px, py), 4, (255, 0, 0), -1)


            if len(trajectory) >= 2:
                pts = np.array(list(trajectory), dtype=np.int32)
                cv2.polylines(annotated, [pts], isClosed=False, color=(0, 0, 255), thickness=2)


            out.write(annotated)

            dataset_rows.append({
                "video": video_path.name,
                "frame": frame_idx,
                "bbox_x1": last_bbox[0] if last_bbox else -1,
                "bbox_y1": last_bbox[1] if last_bbox else -1,
                "bbox_x2": last_bbox[2] if last_bbox else -1,
                "bbox_y2": last_bbox[3] if last_bbox else -1,
                "center_x": px,
                "center_y": py,
                "confidence": last_bbox[4] if last_bbox else 0.0,
                "missed_frames": missed_frames
            })

        frame_idx += 1

    cap.release()
    out.release()
    print(f"Saved: {out_path}")

print("Kalman-tracked videos created in outputs/videos/")
print("Total dataset rows:", len(dataset_rows))

Videos being processed:
- Drone Tracking 1.mp4
- Drone tracking 2.mp4
Saved: outputs/videos/Drone Tracking 1_tracked.mp4
Saved: outputs/videos/Drone tracking 2_tracked.mp4
Kalman-tracked videos created in outputs/videos/
Total dataset rows: 3308


In [36]:
from pathlib import Path
import subprocess

video_dir = Path(CFG["output_videos_dir"])

for video_path in video_dir.glob("*_tracked.mp4"):
    final_path = video_path.with_name(video_path.stem + "_final.mp4")

    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(video_path),
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        str(final_path)
    ]

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

print("ffmpeg re-encoded videos created.")
print(list(video_dir.glob("*_final.mp4")))

Running: ffmpeg -y -i outputs/videos/Drone tracking 2_tracked.mp4 -vcodec libx264 -pix_fmt yuv420p outputs/videos/Drone tracking 2_tracked_final.mp4
Running: ffmpeg -y -i outputs/videos/Drone Tracking 1_tracked.mp4 -vcodec libx264 -pix_fmt yuv420p outputs/videos/Drone Tracking 1_tracked_final.mp4
Running: ffmpeg -y -i outputs/videos/sample_tracked.mp4 -vcodec libx264 -pix_fmt yuv420p outputs/videos/sample_tracked_final.mp4
ffmpeg re-encoded videos created.
[PosixPath('outputs/videos/Drone Tracking 1_tracked_final.mp4'), PosixPath('outputs/videos/sample_tracked_final.mp4'), PosixPath('outputs/videos/Drone tracking 2_tracked_final.mp4')]


In [37]:
import pandas as pd
from pathlib import Path

if "dataset_rows" not in globals():
    dataset_rows = []

if len(dataset_rows) == 0:
    dataset_rows = [{
        "video": "no_detection_fallback.mp4",
        "frame": 0,
        "bbox_x1": 0,
        "bbox_y1": 0,
        "bbox_x2": 100,
        "bbox_y2": 100,
        "center_x": 50,
        "center_y": 50,
        "confidence": 0.0,
        "missed_frames": 0
    }]

Path(CFG["hf_dataset_dir"]).mkdir(parents=True, exist_ok=True)

df = pd.DataFrame(dataset_rows)
parquet_path = str(Path(CFG["hf_dataset_dir"]) / "detections.parquet")
df.to_parquet(parquet_path, index=False)

print("Dataset saved at:", parquet_path)
print(df.head())

Dataset saved at: outputs/hf_dataset/detections.parquet
                  video  frame  bbox_x1  bbox_y1  bbox_x2  bbox_y2  center_x  \
0  Drone Tracking 1.mp4      0      281       58      305       76       293   
1  Drone Tracking 1.mp4      1      281       58      305       76       293   
2  Drone Tracking 1.mp4      2      281       58      305       76       293   
3  Drone Tracking 1.mp4      3      281       58      305       76       293   
4  Drone Tracking 1.mp4      4      281       58      305       76       293   

   center_y  confidence  missed_frames  
0        67    0.765545              0  
1        67    0.765545              0  
2        67    0.765545              0  
3        67    0.765545              0  
4        67    0.765545              0  


In [38]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
from pathlib import Path

print("Videos:", list(Path("outputs/videos").glob("*.mp4")))
print("Dataset:", list(Path("outputs/hf_dataset").glob("*")))

Videos: [PosixPath('outputs/videos/Drone Tracking 1_tracked_final.mp4'), PosixPath('outputs/videos/Drone tracking 2_tracked.mp4'), PosixPath('outputs/videos/Drone Tracking 1_tracked.mp4'), PosixPath('outputs/videos/sample_tracked_final.mp4'), PosixPath('outputs/videos/Drone tracking 2_tracked_final.mp4'), PosixPath('outputs/videos/sample_tracked.mp4')]
Dataset: [PosixPath('outputs/hf_dataset/detections.parquet')]
